# SAC-GRAPH5G: Research-Grade GNN + Reinforcement Learning for 5G Network Data

This notebook loads `5g_network_data.csv` directly from the current notebook folder. It evaluates an advanced residual GraphSAGE model controlled by Soft Actor-Critic (SAC), and it adds the reviewer-critical pieces: strong baselines, random-search and Optuna ablations, multi-seed mean +/- std reporting, richer SAC state, and paper-ready exports.

In [1]:
# Install every external library used by this notebook.
%pip install -q pandas numpy scipy scikit-learn matplotlib seaborn tqdm torch optuna openpyxl


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
import time
import warnings
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__, "Optuna:", optuna.__version__)

Device: cuda
Torch: 2.8.0+cu129 Optuna: 4.9.0


## 1. Configuration

For a paper run, keep 5 seeds or increase to 10. For a quick smoke test, reduce `SEEDS`, `SEARCH_BUDGET`, and the epoch counts.

In [3]:
DATA_FILE = Path("5g_network_data.csv")
TARGET_COLUMN = None

SEEDS = [42]
MAX_FEATURES = 256
SEARCH_BUDGET = 24
SEARCH_EPOCHS = 45
SEARCH_PATIENCE = 12
FINAL_EPOCHS = 180
FINAL_PATIENCE = 30

RUN_RANDOM_SEARCH = True
RUN_OPTUNA_SEARCH = True
RUN_SAC_SEARCH = True

ARTIFACT_DIR = Path("sac_graph5g_outputs")
ARTIFACT_DIR.mkdir(exist_ok=True)

## 2. Load the Local CSV

The dataset is loaded directly from `5g_network_data.csv`. There are no Kaggle API calls in this version.

In [4]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE.resolve()}. Put 5g_network_data.csv beside this notebook "
        "or change DATA_FILE to the correct local CSV path."
    )

df = pd.read_csv(DATA_FILE)
df.columns = [str(c).strip() for c in df.columns]
df = df.drop_duplicates().reset_index(drop=True)
print("Loaded:", DATA_FILE.resolve())
print("Shape:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique(dropna=True)}))

Loaded: /root/5g_network_data.csv
Shape: (50000, 21)


,Timestamp,Location,Signal Strength (dBm),Download Speed (Mbps),Upload Speed (Mbps),Latency (ms),Jitter (ms),Network Type,Device Model,Carrier,...,Battery Level (%),Temperature (°C),Connected Duration (min),Handover Count,Data Usage (MB),Video Streaming Quality,VoNR Enabled,Network Congestion Level,Ping to Google (ms),Dropped Connection
0,2025-05-28 06:59:51.089339,San Francisco,-108.6,714.94,60.41,10.0,4.09,5G NSA,iPhone 14,AT&T,...,99,35.5,14,1,97.40,4,False,High,27.9,True
1,2025-05-28 06:49:51.089353,San Francisco,-71.5,686.69,148.70,12.3,1.50,4G,Pixel 7,AT&T,...,67,22.0,51,4,143.23,3,True,Medium,22.2,False
2,2025-05-28 06:39:51.089356,Chennai,-67.5,796.34,136.33,19.9,1.22,5G NSA,iPhone 14,Airtel,...,77,36.1,45,2,179.15,5,False,Low,75.5,False
3,2025-05-28 06:29:51.089360,New York,-73.3,208.56,68.59,12.2,4.94,4G,Pixel 7,T-Mobile,...,25,39.3,48,0,128.87,4,False,High,87.5,False
4,2025-05-28 06:19:51.089363,Kolkata,-93.2,409.85,137.23,6.3,2.94,5G NSA,Galaxy S23,BSNL,...,51,22.7,54,4,156.91,1,True,Medium,32.5,True


,dtype,missing,unique
Timestamp,object,0,50000
Location,object,0,8
Signal Strength (dBm),float64,0,501
Download Speed (Mbps),float64,0,38384
Upload Speed (Mbps),float64,0,12738
Latency (ms),float64,0,191
Jitter (ms),float64,0,491
Network Type,object,0,3
Device Model,object,0,5
Carrier,object,0,7


In [5]:
TARGET_CANDIDATES = [
    "label", "Label", "LABEL", "class", "Class", "CLASS", "target", "Target", "TARGET",
    "attack", "Attack", "attack_type", "Attack Type", "type", "Type", "slice type",
    "Slice Type", "network type", "Network Type", "traffic type", "Traffic Type",
    "output", "Output", "category", "Category"
]

def infer_target_column(frame):
    for name in TARGET_CANDIDATES:
        if name in frame.columns and 2 <= frame[name].nunique(dropna=True) <= max(50, len(frame) // 2):
            return name
    low_cardinality = [
        c for c in frame.columns
        if 2 <= frame[c].nunique(dropna=True) <= max(20, int(0.2 * len(frame)))
    ]
    return low_cardinality[-1] if low_cardinality else frame.columns[-1]

target_col = TARGET_COLUMN or infer_target_column(df)
df = df.dropna(subset=[target_col]).copy()

if pd.api.types.is_numeric_dtype(df[target_col]) and df[target_col].nunique(dropna=True) > 50:
    print("High-cardinality numeric target detected. Creating quartile classes.")
    df[target_col] = pd.qcut(df[target_col], q=4, labels=["Q1_low", "Q2_midlow", "Q3_midhigh", "Q4_high"], duplicates="drop")

counts = df[target_col].astype(str).value_counts()
rare = counts[counts < 3]
if len(rare):
    print("Removing rare labels with fewer than 3 rows:", rare.to_dict())
    df = df[~df[target_col].astype(str).isin(rare.index)].reset_index(drop=True)

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(df[target_col].astype(str))
class_names = list(label_encoder.classes_)
X_raw_all = df.drop(columns=[target_col])

print("Target column:", target_col)
print("Rows:", len(df))
print("Classes:", class_names)
print("Class counts:", dict(zip(class_names, np.bincount(y_all))))

Target column: Network Type
Rows: 50000
Classes: ['4G', '5G NSA', '5G SA']
Class counts: {'4G': np.int64(16549), '5G NSA': np.int64(16793), '5G SA': np.int64(16658)}


## 3. Leakage-Safe Preprocessing and Graph Construction

For each seed, splitting happens before preprocessing. The preprocessor is fitted on training rows only, then applied to all rows for transductive graph construction without using validation/test labels.

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_preprocessor(raw_frame):
    numeric_cols = raw_frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in raw_frame.columns if c not in numeric_cols]
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    try:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse=True)
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", one_hot)])
    return ColumnTransformer([("num", numeric_pipe, numeric_cols), ("cat", categorical_pipe, categorical_cols)], sparse_threshold=0.3)

def stratified_splits(y, seed):
    idx = np.arange(len(y))
    strat = y if np.min(np.bincount(y)) >= 3 else None
    train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=seed, stratify=strat)
    temp_y = y[temp_idx]
    strat_temp = temp_y if np.min(np.bincount(temp_y)) >= 2 else None
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=strat_temp)
    return train_idx, val_idx, test_idx

def prepare_context(seed):
    set_seed(seed)
    train_idx, val_idx, test_idx = stratified_splits(y_all, seed)
    pre = make_preprocessor(X_raw_all)
    X_train_pre = pre.fit_transform(X_raw_all.iloc[train_idx])
    X_pre = pre.transform(X_raw_all)
    n_features = X_pre.shape[1]
    n_comp = min(MAX_FEATURES, X_train_pre.shape[0] - 1, n_features - 1)
    if n_features > MAX_FEATURES and n_comp >= 2:
        reducer = TruncatedSVD(n_components=n_comp, random_state=seed)
        reducer.fit(X_train_pre)
        X = reducer.transform(X_pre).astype(np.float32)
        svd_var = float(reducer.explained_variance_ratio_.sum())
    else:
        X = X_pre.toarray() if hasattr(X_pre, "toarray") else np.asarray(X_pre)
        X = X.astype(np.float32)
        svd_var = None
    return {
        "seed": seed, "X": X, "y": y_all.astype(np.int64),
        "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
        "x_tensor": torch.tensor(X, dtype=torch.float32, device=DEVICE),
        "y_tensor": torch.tensor(y_all, dtype=torch.long, device=DEVICE),
        "train_idx_t": torch.tensor(train_idx, dtype=torch.long, device=DEVICE),
        "val_idx_t": torch.tensor(val_idx, dtype=torch.long, device=DEVICE),
        "test_idx_t": torch.tensor(test_idx, dtype=torch.long, device=DEVICE),
        "svd_explained_variance": svd_var,
    }

def build_knn_graph(features, k=10, metric="cosine"):
    n = features.shape[0]
    k = int(max(1, min(k, n - 1)))
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric=metric)
    nbrs.fit(features)
    _, neigh = nbrs.kneighbors(features)
    rows = np.repeat(np.arange(n), k)
    cols = neigh[:, 1:].reshape(-1)
    adj = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(n, n))
    adj = adj.maximum(adj.T)
    adj.setdiag(1.0)
    adj = adj.tocoo()
    deg = np.asarray(adj.sum(axis=1)).reshape(-1)
    deg_inv = np.zeros_like(deg, dtype=np.float32)
    deg_inv[deg > 0] = np.power(deg[deg > 0], -0.5)
    vals = adj.data.astype(np.float32) * deg_inv[adj.row] * deg_inv[adj.col]
    idx = torch.tensor(np.vstack([adj.row, adj.col]), dtype=torch.long, device=DEVICE)
    val = torch.tensor(vals, dtype=torch.float32, device=DEVICE)
    return torch.sparse_coo_tensor(idx, val, size=(n, n), device=DEVICE).coalesce()

ctx0 = prepare_context(SEEDS[0])
print("Feature matrix:", ctx0["X"].shape)
print("Split sizes:", {"train": len(ctx0["train_idx"]), "validation": len(ctx0["val_idx"]), "test": len(ctx0["test_idx"])})
print("Example graph edges:", build_knn_graph(ctx0["X"], k=10)._nnz())

Feature matrix: (50000, 256)
Split sizes: {'train': 35000, 'validation': 7500, 'test': 7500}
Example graph edges: 651318


## 4. Residual GraphSAGE Model

In [7]:
class SageLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.self_proj = nn.Linear(in_dim, out_dim)
        self.neigh_proj = nn.Linear(in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adj):
        h = self.self_proj(x) + self.neigh_proj(torch.sparse.mm(adj, x))
        return self.dropout(F.gelu(self.norm(h)))

class ResidualGraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, depth=3, dropout=0.25):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([SageLayer(hidden_dim, hidden_dim, dropout) for _ in range(depth)])
        self.head = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim))

    def forward(self, x, adj):
        h = F.gelu(self.input(x))
        for layer in self.layers:
            h = h + layer(h, adj)
        return self.head(h)

def metric_bundle(y_true, y_pred, probs=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if probs is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, probs[:, 1]) if len(class_names) == 2 else roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
        except Exception:
            out["roc_auc"] = np.nan
    return out

@torch.no_grad()
def evaluate_gnn(model, adj, ctx, split="val"):
    model.eval()
    idx_t = ctx[f"{split}_idx_t"]
    logits = model(ctx["x_tensor"], adj)
    probs = F.softmax(logits[idx_t], dim=1).detach().cpu().numpy()
    pred = probs.argmax(axis=1)
    true = ctx["y_tensor"][idx_t].detach().cpu().numpy()
    out = metric_bundle(true, pred, probs)
    out.update({"y_true": true, "y_pred": pred, "probs": probs})
    return out

def reward_from_metrics(metrics, params):
    complexity = np.mean([params["hidden_dim"] / 256, params["depth"] / 5, params["k"] / 30])
    return float(0.65 * metrics["f1_macro"] + 0.30 * metrics["accuracy"] - 0.03 * complexity)

def train_gnn(ctx, params, max_epochs=120, patience=20, verbose=False):
    set_seed(ctx["seed"])
    adj = build_knn_graph(ctx["X"], k=params["k"])
    model = ResidualGraphSAGE(ctx["X"].shape[1], params["hidden_dim"], len(class_names), params["depth"], params["dropout"]).to(DEVICE)
    counts = np.bincount(ctx["y"][ctx["train_idx"]], minlength=len(class_names))
    weights = counts.sum() / np.maximum(counts, 1)
    weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    best_state, best_reward, best_epoch, history = None, -np.inf, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(ctx["x_tensor"], adj)
        loss = F.cross_entropy(logits[ctx["train_idx_t"]], ctx["y_tensor"][ctx["train_idx_t"]], weight=weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        opt.step()
        val = evaluate_gnn(model, adj, ctx, "val")
        tr = evaluate_gnn(model, adj, ctx, "train")
        history.append({"epoch": epoch, "loss": float(loss.detach().cpu()), "train_accuracy": tr["accuracy"], "val_accuracy": val["accuracy"], "val_f1_macro": val["f1_macro"]})
        score = reward_from_metrics(val, params)
        if score > best_reward:
            best_reward, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch == 1 or epoch % 20 == 0):
            print(f"epoch={epoch:03d} loss={history[-1]['loss']:.4f} val_acc={val['accuracy']:.4f} val_f1={val['f1_macro']:.4f}")
        if epoch - best_epoch >= patience:
            break
    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    return model, adj, pd.DataFrame(history), best_reward

DEFAULT_GNN_PARAMS = {"k": 10, "hidden_dim": 128, "depth": 3, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4}

## 5. Baselines and Search Controllers

Baselines include Logistic Regression, Random Forest, MLP, default GraphSAGE, random-search GraphSAGE, Optuna/TPE GraphSAGE, and the proposed SAC-GRAPH5G.

In [8]:
def run_classical_baselines(ctx):
    X, y = ctx["X"], ctx["y"]
    X_train, y_train = X[ctx["train_idx"]], y[ctx["train_idx"]]
    X_test, y_test = X[ctx["test_idx"]], y[ctx["test_idx"]]
    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "RandomForest": RandomForestClassifier(n_estimators=350, class_weight="balanced_subsample", random_state=ctx["seed"], n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400, random_state=ctx["seed"], early_stopping=True),
    }
    rows = []
    for name, clf in models.items():
        start = time.time()
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        probs = clf.predict_proba(X_test) if hasattr(clf, "predict_proba") else None
        rows.append({"method": name, "seed": ctx["seed"], "seconds": time.time() - start, **metric_bundle(y_test, pred, probs)})
    return rows

def final_train_from_params(ctx, method_name, params):
    start = time.time()
    model, adj, hist, _ = train_gnn(ctx, params, FINAL_EPOCHS, FINAL_PATIENCE)
    test = evaluate_gnn(model, adj, ctx, "test")
    row = {"method": method_name, "seed": ctx["seed"], "seconds": time.time() - start, "selected_params": json.dumps(params), **{k: test[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc"]}}
    return row, hist, test

ACTION_DIM = 6
HIDDEN_OPTIONS = np.array([32, 64, 96, 128, 192, 256])

def decode_action(action):
    u = (np.clip(np.asarray(action, dtype=np.float32), -1, 1) + 1) / 2
    return {
        "k": int(round(3 + u[0] * 27)),
        "hidden_dim": int(HIDDEN_OPTIONS[min(len(HIDDEN_OPTIONS) - 1, int(u[1] * len(HIDDEN_OPTIONS)))]),
        "depth": int(round(2 + u[2] * 3)),
        "dropout": float(0.05 + u[3] * 0.45),
        "lr": float(10 ** (-4.3 + u[4] * 1.8)),
        "weight_decay": float(10 ** (-6.0 + u[5] * 4.5)),
    }

def evaluate_params_short(ctx, params):
    model, adj, hist, _ = train_gnn(ctx, params, SEARCH_EPOCHS, SEARCH_PATIENCE)
    val = evaluate_gnn(model, adj, ctx, "val")
    return reward_from_metrics(val, params), val, hist

def random_search(ctx, budget=SEARCH_BUDGET):
    rng = np.random.default_rng(ctx["seed"] + 1000)
    best, rows = {"reward": -np.inf, "params": None}, []
    for trial in range(1, budget + 1):
        params = decode_action(rng.uniform(-1, 1, ACTION_DIM))
        reward, val, _ = evaluate_params_short(ctx, params)
        if reward > best["reward"]:
            best = {"reward": reward, "params": params}
        rows.append({"controller": "RandomSearch", "seed": ctx["seed"], "trial": trial, "reward": reward, "val_accuracy": val["accuracy"], "val_f1_macro": val["f1_macro"], **params})
        print(f"Random seed={ctx['seed']} trial={trial:02d} reward={reward:.4f} val_acc={val['accuracy']:.4f} val_f1={val['f1_macro']:.4f}")
    return best, pd.DataFrame(rows)

def optuna_search(ctx, budget=SEARCH_BUDGET):
    def objective(trial):
        params = {
            "k": trial.suggest_int("k", 3, 30),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 96, 128, 192, 256]),
            "depth": trial.suggest_int("depth", 2, 5),
            "dropout": trial.suggest_float("dropout", 0.05, 0.50),
            "lr": trial.suggest_float("lr", 5e-5, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 3e-2, log=True),
        }
        reward, val, _ = evaluate_params_short(ctx, params)
        trial.set_user_attr("val_accuracy", val["accuracy"])
        trial.set_user_attr("val_f1_macro", val["f1_macro"])
        return reward
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=ctx["seed"]))
    study.optimize(objective, n_trials=budget, show_progress_bar=False)
    rows = [{"controller": "OptunaTPE", "seed": ctx["seed"], "trial": t.number + 1, "reward": t.value, "val_accuracy": t.user_attrs.get("val_accuracy"), "val_f1_macro": t.user_attrs.get("val_f1_macro"), **t.params} for t in study.trials]
    print(f"Optuna seed={ctx['seed']} best_reward={study.best_value:.4f}")
    return {"reward": study.best_value, "params": study.best_params}, pd.DataFrame(rows)

## 6. Soft Actor-Critic Controller

The SAC state includes optimization history plus dataset signals: class entropy, imbalance, feature dimensionality, split ratios, feature dispersion, and previous action. This is much richer than a toy state of only last reward and step count.

In [9]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        s, a, r, ns, d = map(np.asarray, zip(*batch))
        return (torch.tensor(s, dtype=torch.float32, device=DEVICE), torch.tensor(a, dtype=torch.float32, device=DEVICE), torch.tensor(r, dtype=torch.float32, device=DEVICE).unsqueeze(1), torch.tensor(ns, dtype=torch.float32, device=DEVICE), torch.tensor(d, dtype=torch.float32, device=DEVICE).unsqueeze(1))
    def __len__(self):
        return len(self.buffer)

class SACActor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=192):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, action_dim)
        self.log_std = nn.Linear(hidden, action_dim)
    def sample(self, state):
        h = self.net(state)
        mu = self.mu(h)
        std = self.log_std(h).clamp(-5, 2).exp()
        normal = torch.distributions.Normal(mu, std)
        z = normal.rsample()
        action = torch.tanh(z)
        logp = normal.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        return action, logp.sum(dim=1, keepdim=True)

class SACCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=192):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim + action_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=1))

def class_entropy(y):
    p = np.bincount(y, minlength=len(class_names)).astype(np.float64)
    p = p / p.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(max(len(class_names), 2)))

def sac_state(ctx, ep, budget, last_reward, best_reward, last_acc, last_f1, slope, last_action):
    counts = np.bincount(ctx["y"][ctx["train_idx"]], minlength=len(class_names))
    imbalance = float(counts.max() / max(counts.min(), 1))
    dispersion = float(np.mean(np.std(ctx["X"], axis=0)))
    base = np.array([
        ep / max(budget, 1), last_reward, best_reward, last_acc, last_f1, slope,
        min(len(ctx["y"]) / 10000, 1.0), min(ctx["X"].shape[1] / MAX_FEATURES, 1.0),
        class_entropy(ctx["y"][ctx["train_idx"]]), min(imbalance / 50, 1.0),
        min(dispersion, 5.0) / 5.0, len(ctx["train_idx"]) / len(ctx["y"]), len(ctx["val_idx"]) / len(ctx["y"]),
    ], dtype=np.float32)
    return np.concatenate([base, np.asarray(last_action, dtype=np.float32)])

def soft_update(target, source, tau=0.025):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.mul_(1 - tau).add_(tau * sp.data)

def sac_update(actor, q1, q2, tq1, tq2, actor_opt, q1_opt, q2_opt, replay, batch_size=32, gamma=0.92, alpha=0.12):
    if len(replay) < 8:
        return
    state, action, reward, next_state, done = replay.sample(batch_size)
    with torch.no_grad():
        next_action, next_logp = actor.sample(next_state)
        target_q = torch.min(tq1(next_state, next_action), tq2(next_state, next_action)) - alpha * next_logp
        target = reward + gamma * (1 - done) * target_q
    q1_loss = F.mse_loss(q1(state, action), target)
    q2_loss = F.mse_loss(q2(state, action), target)
    q1_opt.zero_grad(set_to_none=True); q1_loss.backward(); q1_opt.step()
    q2_opt.zero_grad(set_to_none=True); q2_loss.backward(); q2_opt.step()
    new_action, logp = actor.sample(state)
    actor_loss = (alpha * logp - torch.min(q1(state, new_action), q2(state, new_action))).mean()
    actor_opt.zero_grad(set_to_none=True); actor_loss.backward(); actor_opt.step()
    soft_update(tq1, q1); soft_update(tq2, q2)

def sac_search(ctx, budget=SEARCH_BUDGET):
    set_seed(ctx["seed"])
    last_action = np.zeros(ACTION_DIM, dtype=np.float32)
    state = sac_state(ctx, 0, budget, 0, 0, 0, 0, 0, last_action)
    actor = SACActor(len(state), ACTION_DIM).to(DEVICE)
    q1, q2 = SACCritic(len(state), ACTION_DIM).to(DEVICE), SACCritic(len(state), ACTION_DIM).to(DEVICE)
    tq1, tq2 = SACCritic(len(state), ACTION_DIM).to(DEVICE), SACCritic(len(state), ACTION_DIM).to(DEVICE)
    tq1.load_state_dict(q1.state_dict()); tq2.load_state_dict(q2.state_dict())
    actor_opt = torch.optim.Adam(actor.parameters(), lr=3e-4)
    q1_opt = torch.optim.Adam(q1.parameters(), lr=3e-4)
    q2_opt = torch.optim.Adam(q2.parameters(), lr=3e-4)
    replay, rewards, rows = ReplayBuffer(), [], []
    best = {"reward": -np.inf, "params": None}
    warmup = max(6, budget // 4)
    for ep in range(1, budget + 1):
        if ep <= warmup:
            action = np.random.uniform(-1, 1, ACTION_DIM).astype(np.float32)
        else:
            with torch.no_grad():
                action = actor.sample(torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0))[0].squeeze(0).cpu().numpy()
        params = decode_action(action)
        reward, val, _ = evaluate_params_short(ctx, params)
        rewards.append(reward)
        if reward > best["reward"]:
            best = {"reward": reward, "params": params}
        slope = 0.0 if len(rewards) < 6 else float(np.mean(rewards[-3:]) - np.mean(rewards[-6:-3]))
        next_state = sac_state(ctx, ep, budget, reward, best["reward"], val["accuracy"], val["f1_macro"], slope, action)
        replay.add(state, action, reward, next_state, ep == budget)
        for _ in range(10):
            sac_update(actor, q1, q2, tq1, tq2, actor_opt, q1_opt, q2_opt, replay)
        state = next_state
        rows.append({"controller": "SAC-GRAPH5G", "seed": ctx["seed"], "trial": ep, "reward": reward, "val_accuracy": val["accuracy"], "val_f1_macro": val["f1_macro"], **params})
        print(f"SAC seed={ctx['seed']} ep={ep:02d} reward={reward:.4f} val_acc={val['accuracy']:.4f} val_f1={val['f1_macro']:.4f}")
    return best, pd.DataFrame(rows)

## 7. Multi-Seed Experiment

In [10]:
all_results, search_tables, final_histories, final_predictions = [], [], {}, {}

for seed in SEEDS:
    print("\n" + "=" * 90)
    print("Seed:", seed)
    ctx = prepare_context(seed)

    rows = run_classical_baselines(ctx)
    all_results.extend(rows)
    for r in rows:
        print(f"{r['method']}: acc={r['accuracy']:.4f} f1={r['f1_macro']:.4f}")

    row, hist, _ = final_train_from_params(ctx, "Default-GraphSAGE", DEFAULT_GNN_PARAMS)
    all_results.append(row)
    final_histories[(seed, "Default-GraphSAGE")] = hist
    print(f"Default-GraphSAGE: acc={row['accuracy']:.4f} f1={row['f1_macro']:.4f}")

    if RUN_RANDOM_SEARCH:
        best, hist_s = random_search(ctx)
        search_tables.append(hist_s)
        row, hist, _ = final_train_from_params(ctx, "RandomSearch-GraphSAGE", best["params"])
        all_results.append(row)
        final_histories[(seed, "RandomSearch-GraphSAGE")] = hist
        print(f"RandomSearch-GraphSAGE: acc={row['accuracy']:.4f} f1={row['f1_macro']:.4f}")

    if RUN_OPTUNA_SEARCH:
        best, hist_s = optuna_search(ctx)
        search_tables.append(hist_s)
        row, hist, _ = final_train_from_params(ctx, "OptunaTPE-GraphSAGE", best["params"])
        all_results.append(row)
        final_histories[(seed, "OptunaTPE-GraphSAGE")] = hist
        print(f"OptunaTPE-GraphSAGE: acc={row['accuracy']:.4f} f1={row['f1_macro']:.4f}")

    if RUN_SAC_SEARCH:
        best, hist_s = sac_search(ctx)
        search_tables.append(hist_s)
        row, hist, test = final_train_from_params(ctx, "SAC-GRAPH5G", best["params"])
        all_results.append(row)
        final_histories[(seed, "SAC-GRAPH5G")] = hist
        final_predictions[seed] = {"test_idx": ctx["test_idx"], "y_true": test["y_true"], "y_pred": test["y_pred"], "probs": test["probs"], "params": best["params"]}
        print(f"SAC-GRAPH5G: acc={row['accuracy']:.4f} f1={row['f1_macro']:.4f}")

results_df = pd.DataFrame(all_results)
search_history_df = pd.concat(search_tables, ignore_index=True) if search_tables else pd.DataFrame()
display(results_df.sort_values(["method", "seed"]))


Seed: 42
LogisticRegression: acc=0.3392 f1=0.3391
RandomForest: acc=0.3344 f1=0.3022
MLP: acc=0.3279 f1=0.2968
Default-GraphSAGE: acc=0.3321 f1=0.3308
Random seed=42 trial=01 reward=0.3038 val_acc=0.3368 val_f1=0.3367
Random seed=42 trial=02 reward=0.3050 val_acc=0.3411 val_f1=0.3375
Random seed=42 trial=03 reward=0.3078 val_acc=0.3427 val_f1=0.3404
Random seed=42 trial=04 reward=0.2983 val_acc=0.3359 val_f1=0.3331
Random seed=42 trial=05 reward=0.2535 val_acc=0.3307 val_f1=0.2612
Random seed=42 trial=06 reward=0.2987 val_acc=0.3404 val_f1=0.3245
Random seed=42 trial=07 reward=0.2957 val_acc=0.3309 val_f1=0.3301
Random seed=42 trial=08 reward=0.2814 val_acc=0.3292 val_f1=0.3145
Random seed=42 trial=09 reward=0.2996 val_acc=0.3343 val_f1=0.3335
Random seed=42 trial=10 reward=0.2923 val_acc=0.3336 val_f1=0.3238
Random seed=42 trial=11 reward=0.3145 val_acc=0.3428 val_f1=0.3382
Random seed=42 trial=12 reward=0.3113 val_acc=0.3447 val_f1=0.3435
Random seed=42 trial=13 reward=0.2841 val_ac

## 8. Accuracy, Mean +/- Std, and Research Figures

In [11]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc"]
summary_rows = []
for method, g in results_df.groupby("method"):
    row = {"method": method, "n_seeds": g["seed"].nunique()}
    for col in metric_cols:
        row[f"{col}_mean"] = g[col].mean()
        row[f"{col}_std"] = g[col].std(ddof=1)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).sort_values("f1_macro_mean", ascending=False)
display(summary_df)

paper_table = summary_df[["method", "n_seeds"]].copy()
for col in metric_cols:
    paper_table[col] = summary_df.apply(lambda r: f"{r[f'{col}_mean']:.4f} +/- {r[f'{col}_std']:.4f}", axis=1)
display(paper_table)

plt.figure(figsize=(11, 5))
sns.barplot(data=results_df, x="method", y="f1_macro", errorbar="sd")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Test macro-F1")
plt.title("Multi-seed comparison")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.barplot(data=results_df, x="method", y="accuracy", errorbar="sd")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Test accuracy")
plt.title("Multi-seed accuracy comparison")
plt.tight_layout()
plt.show()

if not search_history_df.empty:
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=search_history_df, x="trial", y="reward", hue="controller", errorbar="sd", marker="o")
    plt.title("Equal-budget search-controller comparison")
    plt.tight_layout()
    plt.show()

In [12]:
if final_predictions:
    seed = sorted(final_predictions)[0]
    pack = final_predictions[seed]
    print("Detailed SAC-GRAPH5G test report for seed:", seed)
    print("Selected SAC params:", pack["params"])
    print(classification_report(pack["y_true"], pack["y_pred"], target_names=class_names, zero_division=0))
    cm = confusion_matrix(pack["y_true"], pack["y_pred"])
    plt.figure(figsize=(max(7, len(class_names) * 0.8), max(5, len(class_names) * 0.6)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"SAC-GRAPH5G confusion matrix, seed {seed}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

## 9. Export Paper Artifacts

In [13]:
results_df.to_csv(ARTIFACT_DIR / "all_seed_results.csv", index=False)
summary_df.to_csv(ARTIFACT_DIR / "mean_std_summary.csv", index=False)
paper_table.to_csv(ARTIFACT_DIR / "paper_table.csv", index=False)
search_history_df.to_csv(ARTIFACT_DIR / "controller_search_history.csv", index=False)

for seed, pack in final_predictions.items():
    pred_df = pd.DataFrame({
        "row_index": pack["test_idx"],
        "y_true": [class_names[i] for i in pack["y_true"]],
        "y_pred": [class_names[i] for i in pack["y_pred"]],
    })
    for i, name in enumerate(class_names):
        pred_df[f"prob_{name}"] = pack["probs"][:, i]
    pred_df.to_csv(ARTIFACT_DIR / f"sac_graph5g_test_predictions_seed_{seed}.csv", index=False)

metadata = {
    "data_file": str(DATA_FILE.resolve()),
    "target_column": target_col,
    "rows": int(len(df)),
    "classes": class_names,
    "seeds": SEEDS,
    "search_budget": SEARCH_BUDGET,
    "search_epochs": SEARCH_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "method": "SAC-GRAPH5G: residual GraphSAGE with Soft Actor-Critic graph/training-policy search",
    "baselines": sorted(results_df["method"].unique().tolist()),
}
with open(ARTIFACT_DIR / "experiment_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print("Saved artifacts to:", ARTIFACT_DIR.resolve())
print(json.dumps(metadata, indent=2))

## 10. Related Work, Theory, and Limitations

**Related work.** Graph neural networks such as GCN and GraphSAGE are useful when relational or similarity structure improves prediction beyond independent tabular learning. In 5G network data, records often share behavior through traffic type, slice properties, QoS state, radio/network conditions, and application characteristics. Classical tabular baselines are still essential because many public 5G datasets are not physical graphs by default.

**Why compare SAC to Optuna?** Bayesian/TPE optimization is a strong black-box baseline, so SAC-GRAPH5G should not be accepted merely because it uses reinforcement learning. This notebook compares SAC against random search and Optuna under the same trial budget. SAC is motivated as a sequential controller that conditions on optimization history, class imbalance, feature dimensionality, validation response, reward trend, and previous action.

**Theoretical intuition.** The induced kNN graph imposes a smoothness prior: similar network records exchange information through normalized neighborhood aggregation. Residual GraphSAGE preserves node-specific evidence while using graph context. SAC searches the graph density and model policy using a multi-objective reward that balances macro-F1, accuracy, and complexity.

**Limitations.** The graph is similarity-induced rather than a true 5G topology. If real UE, base-station, flow, or slice edges are available, they should be used or fused with kNN edges. SAC also needs enough trials to learn; tiny budgets reduce it to random exploration. For a submission, run 5-10 seeds, report runtime, include confidence intervals or paired tests, and discuss cases where simpler baselines match the proposed method.